# Notebook 4: Exotic Options

European options depend only on the terminal stock price. Asian and barrier options
depend on the full path, which makes Monte Carlo the natural pricing tool — there is
no closed-form formula to fall back on. This notebook prices both types and verifies
their structural properties numerically: the Asian-European price ordering, barrier
convergence as the barrier recedes to infinity, and the counter-intuitive inverse
relationship between volatility and barrier option prices.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'build'))

import mc_engine
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm

def bs_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return float(S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))

S, r, sigma, T = 100.0, 0.05, 0.20, 1.0
N = 1_000_000
print('Setup complete.')

## Section 1 — Asian Options

An **arithmetic Asian call** has payoff:

$$\max\!\left(\bar{S} - K,\, 0\right), \quad \bar{S} = \frac{1}{n}\sum_{i=1}^{n} S_{t_i}$$

The Asian call is always cheaper than the corresponding European call. Averaging
reduces the effective volatility of the payoff, which lowers option value. More
formally, by Jensen's inequality the averaged path has less dispersion than the
terminal value alone, so the distribution of $\bar{S}$ is tighter than $S_T$.
The effective volatility of $\bar{S}$ is approximately $\sigma/\sqrt{3}$ for uniform
monitoring, hence the lower price.

In [ ]:
K = 100.0

eur = mc_engine.GBMPricer(
    spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
    n_paths=N, n_steps=252, use_gpu=True).price_european_call()

asn = mc_engine.GBMPricer(
    spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
    n_paths=N, n_steps=252, use_gpu=True).price_asian_call()

print(f'European call price : {eur.price:.4f}  (SE={eur.standard_error:.4f})')
print(f'Asian    call price : {asn.price:.4f}  (SE={asn.standard_error:.4f})')
print(f'Difference          : {eur.price - asn.price:.4f}')
assert asn.price < eur.price, 'FAIL: Asian price should be < European price!'
print('PASS: Asian < European.')

In [ ]:
# Effect of averaging frequency (n_steps)
step_counts = [12, 52, 252]

hdr = f"{'n_steps':>8} | {'Asian Price':>12} | {'European Price':>14} | {'Difference':>12}"
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)

asian_prices_steps = []
for ns in step_counts:
    p_asn = mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=ns, use_gpu=True).price_asian_call()
    p_eur = mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=ns, use_gpu=True).price_european_call()
    asian_prices_steps.append(p_asn.price)
    diff = p_eur.price - p_asn.price
    print(f"{ns:>8} | {p_asn.price:>12.4f} | {p_eur.price:>14.4f} | {diff:>12.4f}")
print(sep)
print('\nMore averaging steps → price converges as averaging becomes more continuous.')

## Section 2 — Barrier Options: Up-and-Out Call

An **up-and-out call** has the same payoff as a European call **unless** the stock
ever exceeds a barrier level $B > S_0$ during the option's life, in which case the
option is knocked out and pays zero:

$$V_T = \max(S_T - K, 0) \cdot \mathbf{1}\{\max_{0 \le t \le T} S_t < B\}$$

As $B \to \infty$, the barrier is never reached and the up-and-out call converges
to the European call:

$$\lim_{B \to \infty} C_{\text{barrier}}(B) = C_{\text{European}}$$

For any finite $B$, the barrier can only reduce (never increase) the payoff, so
$C_{\text{barrier}}(B) \le C_{\text{European}}$ holds universally. Both predictions
are straightforward to verify with Monte Carlo by sweeping over barrier levels.

In [ ]:
barriers     = [110.0, 120.0, 130.0, 150.0, 200.0, 500.0]
barrier_prices = []

eur_ref = mc_engine.GBMPricer(
    spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
    n_paths=N, n_steps=252, use_gpu=True).price_european_call()

print(f'European reference: {eur_ref.price:.4f}')
print()

for B in barriers:
    res = mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=252, use_gpu=True).price_barrier_call(
            barrier=B, barrier_type=mc_engine.BarrierType.UP_AND_OUT)
    barrier_prices.append(res.price)
    flag = 'OK' if res.price <= eur_ref.price + 3 * eur_ref.standard_error else 'FAIL'
    print(f'  B={B:>5.0f}: price={res.price:.4f}  [{flag}]')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(barriers, barrier_prices, 'bo-', linewidth=2, markersize=7,
        label='Up-and-Out Call')
ax.axhline(eur_ref.price, color='red', linestyle='--', linewidth=1.8,
           label=f'European Call ({eur_ref.price:.3f})')
ax.set_xlabel('Barrier Level B', fontsize=13)
ax.set_ylabel('Call Price', fontsize=13)
ax.set_title('Up-and-Out Call: Convergence to European as B → ∞\n'
             '(S=100, K=100, σ=0.20, T=1yr)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print('\nBarrier price monotonically increases toward European price as B grows.')

## Section 3 — Barrier Option Sensitivity to Volatility

For a **standard European call**, higher volatility always increases the price
(positive Vega).  For an **up-and-out call**, the relationship is **inverted**:

Higher $\sigma$ → stock is more likely to hit the barrier → option is more likely to
be knocked out → **lower** price.

This makes barrier options very sensitive to the volatility model — an important
practical consideration when using Heston vs Black-Scholes.

In [ ]:
sigmas  = [0.10, 0.15, 0.20, 0.30, 0.40]
B_fixed = 120.0
K_fix   = 100.0

barrier_vol_prices  = []
european_vol_prices = []

for sig in sigmas:
    p_bar = mc_engine.GBMPricer(
        spot=S, strike=K_fix, rate=r, volatility=sig, maturity=T,
        n_paths=N, n_steps=252, use_gpu=True).price_barrier_call(
            barrier=B_fixed, barrier_type=mc_engine.BarrierType.UP_AND_OUT)
    p_eur = mc_engine.GBMPricer(
        spot=S, strike=K_fix, rate=r, volatility=sig, maturity=T,
        n_paths=N, n_steps=252, use_gpu=True).price_european_call()
    barrier_vol_prices.append(p_bar.price)
    european_vol_prices.append(p_eur.price)
    print(f'  σ={sig:.2f}: Barrier={p_bar.price:.4f}, European={p_eur.price:.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sigmas, european_vol_prices, 'b-o', linewidth=2, markersize=6, label='European call')
ax.plot(sigmas, barrier_vol_prices,  'r-s', linewidth=2, markersize=6,
        label=f'Up-and-Out call  (B={B_fixed})')
ax.set_xlabel('Volatility σ', fontsize=13)
ax.set_ylabel('Call Price', fontsize=13)
ax.set_title('European vs Up-and-Out Call: Price vs Volatility\n'
             '(S=100, K=100, B=120, T=1yr)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print('\nEuropean: price increases with σ (standard Vega effect).')
print('Barrier:  price decreases with σ (higher vol → more likely to knock out).')